# Project A (ShapeStream): starter notebook

HY562 Fall 2026. This notebook shows how to read the change stream, where your engine plugs in, and how to evaluate it.
Run it from the `kit` folder of Project A inside the course container (`04_Environment`).

Steps:
1. build the toy data (once): see `README.md`, section *Quick start*;
2. read the stream (files or Kafka);
3. implement your engine in `process_batch` and `current_stats`;
4. write the statistics at the end of every epoch and evaluate them against the gold.

In [ ]:
import glob, json, os, sys, time
sys.path.insert(0, os.getcwd())          # the kit folder, so that `shapestream` is importable

from pyspark.sql import functions as F
from shapestream.spark_io import get_spark, EVENT_SCHEMA
from shapestream.common import RDF_TYPE, UNTYPED, IRI_ANY, write_stats
from shapestream import evaluate as ev

spark = get_spark("projectA")
DATA = "data/toy"                 # later: data/dbpedia (built with SETUP.md)
STREAM = f"{DATA}/streams/full"   # Team A4: f"{DATA}/streams/masked30"
TEAM, VARIANT = "A1", "exact"     # change to your team and variant
OUT = f"runs/{TEAM}/{os.path.basename(DATA)}"
os.makedirs(OUT, exist_ok=True)
print(json.load(open(f"{STREAM}/manifest.json")))

## 1. Look at the events
Each event is one triple insertion (`op = '+'`) or deletion (`op = '-'`). Epoch 0 loads snapshot 0; epoch k changes snapshot k-1 into snapshot k.

In [ ]:
events0 = spark.read.schema(EVENT_SCHEMA).json(f"{STREAM}/epoch=0")
events0.show(5, truncate=60)
spark.read.schema(EVENT_SCHEMA).json(f"{STREAM}/epoch=1").groupBy("op").count().show()

## 2. Your engine

Your engine keeps **state** that is updated by every micro-batch of events and can produce the statistics at any time.
The statistics format is described in `shapestream/common.py`:

* `classes[C]` = number of entities of class C (entities without a type count under `UNTYPED`);
* `properties[(C, p)] = (support, min_count, max_count)`;
* `constraints[(C, p, T)] = support`, where T is the class of an IRI object, the datatype of a literal, or `IRI_ANY` for an untyped IRI.

Things to think about before you code (they come up in the checkpoint 1 discussion):
* a deletion can lower a `max_count`; what do you need to keep to recompute it?
* when an entity gets a new type, all its property counts move to the new class;
* when an **object** gets a new type, the constraints of the subjects that point to it change;
* what part of the state must live in Spark (it does not fit in the driver for DBpedia) and what can live in the driver?

The skeleton below only counts events, so that the notebook runs end to end. Replace it.

In [ ]:
class Engine:
    """Replace with your exact engine (checkpoint 1) and then with your variant."""
    def __init__(self):
        self.n_events = 0
        # TODO: your state (DataFrames written to Parquet, Python dicts, sketches, ...)

    def process_batch(self, batch_df, batch_id):
        # batch_df has the EVENT_SCHEMA columns; events of one subject are in seq order
        self.n_events += batch_df.count()
        # TODO: update the state

    def current_stats(self):
        # TODO: return (classes, properties, constraints) as in shapestream/common.py
        return {}, {}, {}

    def state_mb(self):
        # TODO: size of your state in MB (files on disk, or an estimate for in-memory structures)
        return None

engine = Engine()

## 3. Run the stream epoch by epoch (file source)

`trigger(availableNow=True)` processes everything that is in the folder and stops, which gives one measured run per epoch.
`maxFilesPerTrigger` / `maxBytesPerTrigger` control the micro-batch size.

In [ ]:
run_info = {}
epochs = sorted(glob.glob(f"{STREAM}/epoch=*"), key=lambda p: int(p.rsplit("=", 1)[1]))
for ep_dir in epochs:
    k = int(ep_dir.rsplit("=", 1)[1])
    t0 = time.time(); n0 = engine.n_events
    q = (spark.readStream.schema(EVENT_SCHEMA).json(ep_dir)
            .writeStream.foreachBatch(engine.process_batch)
            .option("checkpointLocation", f"{OUT}/_chk/epoch={k}")
            .trigger(availableNow=True).start())
    q.awaitTermination()
    classes, props, cons = engine.current_stats()
    dt = time.time() - t0
    write_stats(f"{OUT}/epoch{k}.json", {"team": TEAM, "variant": VARIANT, "epoch": k, "dataset": os.path.basename(DATA)},
                classes, props, cons)
    n = engine.n_events - n0
    run_info[str(k)] = {"latency_s": round(dt, 3), "events_per_s": round(n / dt) if dt else None, "state_mb": engine.state_mb()}
    print(f"epoch {k}: {n} events in {dt:.1f}s")
json.dump(run_info, open(f"{OUT}/run_info.json", "w"), indent=1)

## 4. Evaluate against the gold
The same evaluation from the command line: `python -m shapestream.evaluate --gold ... --pred ... --run-info ... --results results/results_A.csv`.

In [ ]:
for ep_dir in epochs:
    k = int(ep_dir.rsplit("=", 1)[1])
    meta, rows, extra = ev.evaluate(f"{DATA}/gold/epoch{k}.json", f"{OUT}/epoch{k}.json")
    print(f"epoch {k}", extra)
    for r in rows:
        print("   ", r)

## 5. Kafka source (needed for the latency and throughput measurements)

In a terminal of the container: `python -m shapestream.replay_to_kafka --stream data/toy/streams/full --topic kg-changes --pause 30`.
Every partition receives a `checkpoint` message at the end of each epoch; write the statistics when you have seen it on all partitions.

In [ ]:
from pyspark.sql import types as T
kafka_events = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", os.environ.get("KAFKA_BOOTSTRAP", "kafka:9092"))
    .option("subscribe", "kg-changes").option("startingOffsets", "earliest").load()
    .select(F.col("partition"), F.from_json(F.col("value").cast("string"), EVENT_SCHEMA).alias("e"))
    .select("partition", "e.*"))
# kafka_events.writeStream.foreachBatch(...).start()   # same engine; handle op == 'checkpoint' in process_batch

## 6. SHACL output
`python -m shapestream.to_shacl --stats runs/A1/toy/epoch2.json --out runs/A1/toy/shapes_epoch2.ttl --min-support 10 --min-confidence 0.1`